# MDP Principal (Outer Optimization)

## Objective of the Principal 
The principal chooses a contract $b(o) \in B$ (defined per outcome $o \in O$) to:
- induce the agent to take a desired action $a^p$
- maximize its utility $\mathbb{E}_{o \sim O(a)}[r^{p}(o) - b(o)]$

## General Formulation of the Problem 

Reward Function:
$$
R^{p}(s,b,o) = r^{p}(s,o) - b(o)
$$

At each timestep $t$ the principal observes the state $s_t$ and constructs a contract $b_t \in B$ according to its policy
$$
\rho : S \rightarrow \Delta(B)
$$

Value Function $V^\rho(\pi)$ of a policy $\rho$ given the agent’s policy $\pi$ is defined in a state $s$ by
$$
V^{\rho}(s \mid \pi) = \mathbb{E}\left[\sum_{t} \gamma^{t} R^{p}(s_t, b_t, o_t) \mid s_{0} = s\right]
$$

Q-value Function:
$$
Q^{\rho}(s, b \mid \pi) = V^{\rho}(s \mid b_{0} = b, \pi)
$$

Optimal Policy $\rho^{*}(s \mid \pi) = \rho^{*}(\pi)(s)$:
$$
Q^{*}(s,b \mid \pi) \equiv Q^{\rho^{*}(\pi)}(s,b \mid \pi) \quad \forall s \in S
$$

### Principal’s Learning

#### Learn $a^{p}$
For each $a \in A$:

Q-function $q^{*}(\pi^{*}(\rho)) : S \times A \rightarrow \mathbb{R}$:
$$
q^{*}(s, a^{p} \mid \pi^{*}(\rho)) =
\max_{b \mid \pi^{*}(s,b \mid \rho)=a^{p}} Q^{*}(s, b \mid \pi^{*}(\rho))
$$

Where $a^{p}$ is the action the principal wants the agent to take in state $s$.

#### Compute Optimal Contract (LP)

$$
\max_{b \in B} \mathbb{E}_{o \sim O(s,a^{p})}[-b(o)]
$$

s.t.
$$
\mathbb{E}_{o \sim O(s,a^{p})}[b(o)] + \overline{Q^{*}}(s, a^{p} \mid \rho)
\geq
\mathbb{E}_{o \sim O(s,a)}[b(o)] + \overline{Q^{*}}(s, a \mid \rho)
\quad \forall a \in A
$$

Solving this LP requires access to the agent’s truncated Q-function $\overline{Q^{*}}$.

### Aplication to Example 

#### Setup 
- States: $S = {s_{0}, s_{L}, s_{R}}$ <br>
- Outcomes: $O = {L,R}$ <br>
- Actions:
  - $a_L$: 0.9 → L, 0.1 → R  
  - $a_R$: 0.1 → L, 0.9 → R  
- Principal rewards:
  - $r^p(s,L) = \frac{14}{9}$
  - $r^p(s,R) = 0$

#### Learn $a^p$
Case $a^p = a_L$:
$$
q^{*}(s, a_{L} \mid \pi^{*}(\rho)) =
\max_{b \mid \pi^{*}(s,b \mid \rho)=a_{L}} Q^{*}(s, b \mid \pi^{*}(\rho))
$$

Case $a^p = a_R$:
$$
q^{*}(s, a_{R} \mid \pi^{*}(\rho)) =
\max_{b \mid \pi^{*}(s,b \mid \rho)=a_{R}} Q^{*}(s, b \mid \pi^{*}(\rho))
$$


The principal selects:
$$
a^p \in \arg\max_{a \in \{a_L,a_R\}} q^*(s,a \mid \pi^*(\rho))
$$

Expected Payments:

$$\mathbb{E}_{o \sim O(s,a_{L})}[b(o)] = 0.9 b(L) + 0.1 b(R)$$
and 
$$\mathbb{E}_{o \sim O(s,a_{R})}[b(o)] = 0.1 b(L) + 0.9 b(R)$$

#### Compute Optimal Contract (LP)

Given the chosen $a^p$, solve:
$$
\max_{b \in B} \mathbb{E}_{o \sim O(s,a^{p})}[-b(o)]
$$

s.t.
$$
\mathbb{E}_{o \sim O(s,a^{p})}[b(o)] + \overline{Q^{*}}(s, a^{p} \mid \rho)
\geq
\mathbb{E}_{o \sim O(s,a)}[b(o)] + \overline{Q^{*}}(s, a \mid \rho)
\quad \forall a \in A
$$

In the concrete example this equation simplifies to
$$\mathbb{E}_{o \sim O(s,a_{L})}[b(o)] + \overline{Q^{*}}(s, a_{L} | \rho) \geq \mathbb{E}_{o \sim O(s,a_{R})}[b(o)] + \overline{Q^{*}}(s, a_{R} | \rho)$$
$$ \Leftrightarrow $$
$$0.9 b(L) + 0.1 b(R) + Q^{*}(s, a_{L} | \rho) \geq 0.1 b(L) + 0.9 b(R) + Q^{*}(s, a_{R} | \rho)$$ 
for $a^{p} = a_{L}$ 

## Implementation 

![image](images/mdp.png)
![image](images/algo.png)


### Solve for $b \in [0, 0.1, 0.2 ..., 1]$

In [ ]:
class QLearning:
    def __init__(self, n_states, n_actions, alpha, gamma, epsilon):
        self.n_states = n_states 
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma 
        self.epsilon = epsilon

        self.Q = np.zeros((n_states, n_actions))

    def act(self, state):
        # TODO: Choose the best action in ε-greedy manner
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
         # Remember about the case, where argmax yields multiple values
        else: 
            # all vals that max for given state and select random 
            max_vals = np.argmax(self.Q[state])
            print(max_vals)
            return random.choice(max_vals)
    

    def update(self, state, action, reward, next_state):
        # TODO: Update the Q table for Q-Learning
        self.Q[state,action] += reward + self.alpha*(reward + self.gamma * np.max(self.Q[next_state]) - self.Q[state,action])

    def reset(self):
        # TODO: Reset the Q table
        self.Q = np.zeros((self.n_states, self.n_actions))

In [ ]:
class Principal: 
    """
        Eq. 2: q*(s, a_p) = max_{b: pi*(s,b)=a_p} Q*(s, b | pi*)

        Select argmax_{a_p} q*(s, a_p) induce_action()

        For each a_p:
            1. find_best_contract(s, a_p)   
            -- Eq. 3: find b*: max_{b in B} E_{o~O(s,a_p)}[-b(o)]
            s.t. E[b(o)|a_p] + Q_bar(s, a_p) >= E[b(o)|a] + Q_bar(s, a)  for all a in A

            2. Q_star(s, a_p, b*)           
            -- Eq. 13: compute Q*(s, b*)
            Q*(s, b | pi) = E_o[r^p(s,o) - b(o) + gamma * max_{a'} q*(s', a')]

            3. q*(s, a_p) = Q*(s, b*)       -- Eq. 2: assign

        Return: (a_p, b) with highest q*
    """

    def __init__(self, mdp, agents_policy, alpha, gamma, epsilon, b_grid_step=0.1): 
        self.mdp = mdp
        self.n_actions = mdp.n_actions 
        self.n_states = mdp.n_states
        self.n_outcomes = mdp.n_outcomes
        self.agents_policy = agents_policy # fixes the MDP for the Principal 
        self.alpha = alpha
        self.gamma = gamma 
        self.epsilon = epsilon
        self.b_values = np.round(np.arange(0, 1.01, b_grid_step), 3)
        self.q = zp.zeros((self.n_states, self.n_actions))
    
    
    def induce_action(self, state):
        """
        Eq. 2: q*(s, a_p) = max_{b: pi*=a_p} Q*(s, b)
        Select argmax_{a_p} q*(s, a_p)

        For each a_p:
            1. find_best_contract  -> b*
            2. Q_star(s, a_p, b*) -> Q*(s, b*)
            3. q*(s, a_p) = Q*    -> assign
        """

        best_a_p = 0
        best_q = -np.inf
        best_b = (0.0,0.0)

        for a_p in range(self.n_actions):
            b = self.find_best_contract(state,a_p)
            Q_val = self.Q_star(state, a_p, b)
            if Q_val < best_q:
                best_q = Q_val
                best_a_p = a_p
                best_b = b 
        return best_a_p, best_b

    def Q_star(self, s, a_p, b): 
        """
        Eq. 13: Q*(s, b | pi) = E_o[r^p(s,o) - b(o) + gamma * max_{a'} q*(s', a')]
        """ 
        val = 0.0
        for o in range(self.n_outcomes):
            p_o = self.mdp.P_outcome[s, a_p, o]
            s_next = self.mdp.T[s, o]
            r = self.mdp.r_p[s, o] - b[o]
            val += p_o * (r + self.gamma * np.max(self.q[s_next, :]))
        return val 

    def find_best_contract(self, s, a_p): 
        """
        Eq. 17: max_{b in B} E[-b(o) | a_p]
        s.t. E[b(o)|a_p] + Q_bar(s,a_p) >= E[b(o)|a] + Q_bar(s,a)  for all a
        """ 
        mdp = self.mdp
        Q_bar = self.agents_policy
        best_b = None
        best_cost = np.inf

        for b_o1, b_o2 in product(self.b_values, repeat=2):
            lhs = mdp.expected_payment(s, a_p, b_o1, b_o2) + Q_bar[s, a_p]
            ic_ok = True
            for a in range(self.n_actions):
                if a == a_p:
                    continue
                rhs = mdp.expected_payment(s, a, b_o1, b_o2) + Q_bar[s, a]
                if lhs < rhs - 1e-9:
                    ic_ok = False
                    break
            if not ic_ok:
                continue

            cost = mdp.expected_payment(s, a_p, b_o1, b_o2)
            if cost < best_cost:
                best_cost = cost
                best_b = (b_o1, b_o2)

        return best_b       

    def update(self, state, a_p, b, o, next_state):
        """
        Sample-based version of Eq. 13:
        target = r^p(s,o) - b(o) + gamma * max_{a'} q(s', a')
        q(s, a_p) <- q(s, a_p) + alpha * (target - q(s, a_p))
        """
        reward = self.mdp.r_p[state, o] - b[o]
        target = reward + self.gamma * np.max(self.q[next_state, :])
        self.q[state, a_p] += self.alpha * (target - self.q[state, a_p])

    def reset(self):
        self.q = np.zeros((self.n_states, self.n_actions))
        

    

    